# Sample Size Computations

In this section we will be determining the sample size needed to reach a desired power given a significance level and effect size.

We will be using data from the Michelson speed of light dataset. We will begin with power of $0.8$.

In [ ]:
import micropip

await micropip.install("ipywidgets")

In [ ]:
import numpy as np
import ipywidgets as widgets
from ipywidgets import interactive, fixed
import statsmodels.stats.power as smp
import pandas as pd

We will use a pre-sample of size 10 to determine the required sample size. Additionally we will use statsmodels [`TTestIndPower`](https://www.statsmodels.org/stable/generated/statsmodels.stats.power.TTestIndPower.html) to solve for the sample size needed given the other parameters. Run the code below to read our data.

In [ ]:
data = pd.read_csv("data/morley.csv")

```{note}
When using methods from python modules such as `statsmodel`, it is important to read documentation carefully because there are many different notions of things like "effect size" for instance.
```

Similar computations as the ones conducted in this section can also be performed for other types of hypothesis tests. Adjust the sliders for power, significance level, and effect size and observe the effect on the sample size.

In [ ]:
def determine_sample_size(effect, power, significance, pre_sample_size, true_std):
    # Get a pre-sample to estimate the effect size
    group1 = data["Speed"].sample(n=pre_sample_size)
    group2 = data["Speed"].sample(n=pre_sample_size)

    m1, m2 = group1.mean(), group2.mean()
    s1, s2 = group1.std(ddof=1), group2.std(ddof=1)
    n1 = n2 = pre_sample_size
    pooled_sd = (
        np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
        if (n1 + n2 - 2) > 0
        else np.nan
    )
    print(
        f"Pre-sample (n={pre_sample_size} per group): m1={m1:.4f}, m2={m2:.4f}, pooled_sd={pooled_sd:.4f}"
    )

    # Calculate the required sample size for the given parameters for two sided test
    power_analysis = smp.TTestIndPower()
    sample_size = power_analysis.solve_power(
        effect_size=effect, alpha=significance, power=power, alternative="two-sided"
    )
    print(f"Two-Sided calculated sample size: {sample_size:.2f}")

    # Calculate the required sample size for the given parameters for one sided test
    power_analysis = smp.TTestIndPower()
    sample_size = power_analysis.solve_power(
        effect_size=effect, alpha=significance, power=power, alternative="larger"
    )
    print(f"One-sided calculated sample size: {sample_size:.2f}")

    return sample_size


effect_val = widgets.FloatSlider(
    value=1.5, min=0.1, max=3, step=0.1, description="Effect Size"
)
power_val = widgets.FloatSlider(
    value=0.8, min=0.1, max=0.99, step=0.01, description="Power"
)
significance_val = widgets.FloatSlider(
    value=0.05, min=0.01, max=0.1, step=0.01, description="Significance Level"
)
n = 10
std = np.std(data)

interactive_plot = interactive(
    determine_sample_size,
    effect=effect_val,
    power=power_val,
    significance=significance_val,
    pre_sample_size=fixed(n),
    true_std=fixed(std),
)
interactive_plot